# 1. What you'll learn

        - compare six families on identical folds
- plot repeated-CV uncertainty rather than a single score
- select once and inspect one untouched test

        **The one question this notebook answers:** Which of six model families performs best under one honest protocol, and are the differences larger than uncertainty?

# 2. Setup

This lesson keeps model comparison capstone small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split,RepeatedStratifiedKFold,cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier,HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score,balanced_accuracy_score

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

The breast-cancer dataset supplies a realistic small tabular classification problem. Twenty percent is locked; repeated stratified CV on development compares all families by ROC AUC.

**Small example:** If five fold AUCs are [.91,.95,.93,.89,.94], the mean is .924 but the spread matters. A rival at .928 is not clearly better without paired uncertainty.

In [ ]:
b=load_breast_cancer(as_frame=True);X=b.data;y=(b.target==0).astype(int);print("Shape:",X.shape,"malignant share:",round(y.mean(),3),"features:",X.shape[1]);print(X.iloc[:3,:5].round(2))

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.7));y.value_counts().plot.bar(ax=axes[0],color="#176b66");axes[0].set_title("Outcome counts")
axes[1].hist(X.std(),bins=20,color="#d38b45");axes[1].set_title("Feature-scale spread");sns.heatmap(X.iloc[:,:12].corr(),cmap="vlag",center=0,xticklabels=False,yticklabels=False,ax=axes[2]);axes[2].set_title("Correlated predictors");plt.tight_layout();plt.show()

**Takeaway:** Stratification is needed for the minority malignant class.

**Takeaway:** Distance and linear families require scaling; tree families do not.

**Takeaway:** Correlation favours regularization and makes small score differences unstable.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
X_dev,X_test,y_dev,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=SEED);cv=RepeatedStratifiedKFold(n_splits=5,n_repeats=4,random_state=SEED);models={"logistic":make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000,random_state=SEED)),"naive Bayes":GaussianNB(),"k-NN":make_pipeline(StandardScaler(),KNeighborsClassifier(15,weights="distance")),"RBF SVM":make_pipeline(StandardScaler(),SVC(C=2,probability=True,random_state=SEED)),"random forest":RandomForestClassifier(n_estimators=180,min_samples_leaf=3,n_jobs=1,random_state=SEED),"hist boosting":HistGradientBoostingClassifier(max_iter=120,max_leaf_nodes=15,learning_rate=.06,random_state=SEED)};print("Development/test:",X_dev.shape,X_test.shape,"families:",len(models))

# 6. Train

        Training turns the assumptions behind model comparison capstone into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Lock one final test split before comparison.
2. Evaluate every family on identical repeated stratified folds.
3. Compare paired score distributions and confidence intervals.
4. Select by development mean, refit once, and open the test exactly once.

In [ ]:
rows=[];fits={}
for name,model in models.items():
 out=cross_validate(model,X_dev,y_dev,cv=cv,scoring={"auc":"roc_auc","balanced":"balanced_accuracy"},n_jobs=1,return_train_score=False);rows.extend([{"model":name,"fold":j,"auc":a,"balanced":ba} for j,(a,ba) in enumerate(zip(out["test_auc"],out["test_balanced"]))]);fits[name]=model.fit(X_dev,y_dev)
results=pd.DataFrame(rows);summary=results.groupby("model").auc.agg(["mean","std","count"]).sort_values("mean",ascending=False);summary["se"]=summary["std"]/np.sqrt(summary["count"]);winner=summary.index[0];print(summary.round(4));print("Selected on development:",winner)

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
test_auc={name:roc_auc_score(y_test,m.predict_proba(X_test)[:,1]) for name,m in fits.items()};win_prob=fits[winner].predict_proba(X_test)[:,1];baseline=.5
fig,axes=plt.subplots(1,3,figsize=(15,3.9));order=summary.index[::-1];axes[0].barh(order,summary.loc[order,"mean"],xerr=1.96*summary.loc[order,"se"],color="#176b66");axes[0].axvline(.5,ls="--",color="black");axes[0].set(title="Repeated-CV AUC · 95% mean CI",xlim=(.45,1))
sns.boxplot(data=results,y="model",x="auc",order=order,ax=axes[1],color="#d7e8e4");axes[1].set_title("Fold-to-fold uncertainty")
axes[2].bar(test_auc.keys(),test_auc.values(),color="#d38b45");axes[2].tick_params(axis="x",rotation=70);axes[2].axhline(.5,ls="--",color="black");axes[2].set_ylim(.45,1);axes[2].set_title("Untouched test AUC");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print("Test AUC:",{k:round(v,3) for k,v in test_auc.items()},"selected winner:",winner)

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Single-split leaderboards select sampling luck; across folds the winning family changes and tiny mean gaps may be smaller than uncertainty. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
winners=results.loc[results.groupby("fold").auc.idxmax(),"model"].value_counts();single_best=results.groupby("model").auc.max().sort_values(ascending=False);print("Winner count across 20 resamples:",winners.to_dict());print("Misleading best single-fold score:",single_best.head(3).round(3).to_dict());print("The best one-split model is not a stable leaderboard.")

**Use this when…** you must choose among plausible families under a shared metric, folds, preprocessing boundary, and compute budget.

        **Don't use this when…** you plan to crown a winner from one split, ignore deployment cost, or repeatedly inspect the final test.

        ## Try this

        1. Switch the primary metric to PR-AUC.
2. Add inference latency and model size to the leaderboard.
3. Run nested tuning inside each outer fold.